This notebook demonstrates Poisson reduced-rank regression (p-RRR) on a real electrophysiology dataset. For details on how the model works, see the pRRR_demo_synthetic notebook.

It should take less than 5 mins to run the entire notebook.

## Imports

In [ ]:
import os
import sys 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import time
from mpl_toolkits.axes_grid1 import make_axes_locatable
import sklearn.metrics
from tabulate import tabulate

sys.path.append('../poissonrrr')

from poisson_rrr import *
from linear_rrr import *

## Load data

This is a snippet of ~15 mins (binned at 35 Hz) of an electrophysiology recording from Li, Lu et al, 2025. This experiment used two four-shank Neuropixels probes, one in primary visual cortex (V1) and one in anterior cingulate area (ACA). There are three matrices in this data file:

- `timestamps`: shape (32307), experiment timestamps in seconds
- `v1`: shape (220, 32307), binned spikes from V1 neurons corresponding to timestamps
- `aca`: shape (122, 32307), binned spikes from ACA neurons corresponding to timestamps

In [ ]:
data = np.load('data/demo_data.npz')
aca = data['aca']
v1 = data['v1']
timestamps = data['timestamps']

### Let's visualize our data to see what we're working with

In [ ]:
f = plt.figure(figsize=(16, 5))
plt.plot(timestamps, np.mean(v1, axis=0), c='dodgerblue', label='V1')
plt.plot(timestamps, np.mean(aca, axis=0), c='r', label='ACA')
plt.xlim(2600, 2640)
plt.xlabel('time (s)')
plt.ylabel('mean population activity (spikes/bin)')
plt.legend()

## Use the pRRR model to predict V1 neurons from ACA neurons
~ 4 mins

- We use rank 20, which captures the top 20 shared dimensions between ACA and V1
- Regularization is kept minimal (1e-3). The softplus activation ensures predicted spike counts are non-negative — this is one key advantage of pRRR over linear RRR for count data

In [ ]:
# Arrange data as (samples, features): time bins are samples.
X = aca.T  # ACA predictors
Y = v1.T   # V1 targets

print(f"X shape (samples, n_input): {X.shape}")
print(f"Y shape (samples, n_output): {Y.shape}")

# Train/test split.
train_fraction = 0.8
split_idx = int(X.shape[0] * train_fraction)
X_train, Y_train = X[:split_idx], Y[:split_idx]
X_test, Y_test = X[split_idx:], Y[split_idx:]

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"Y_train: {Y_train.shape}, Y_test: {Y_test.shape}")

# Fit p-RRR: use ACA to predict V1 spike counts.
seed = 1
model_rank = 20
reg_strength = 1e-3

pRRR = PoissonRRR(
    n_input=X_train.shape[1],
    n_output=Y_train.shape[1],
    act="softplus",
    rank=model_rank,
    loss="poisson",
    zeromeanXs=[0],
    normstdXs=[0],
    regList=[reg_strength],
    seed=seed,
)

start = time.time()
train_loss_hist, grad_hist = pRRR.train(
    [X_train],
    Y_train,
    lr=1,
    maxepoch=50,
    max_iter=20,
    history_size=10,
    grad_tol=1e-4,
    line_search=1,
    shuffle=True,
    track=10,
    patience=10,
    loss_tol=1e-6,
    progfile=None,
)
elapsed_time = time.time() - start
print(f"elapsed {elapsed_time:.2f}s")

In [ ]:
# Quick convergence check
fig, ax = plt.subplots(figsize=(4, 2.5))
ax.plot(train_loss_hist, "-o")
ax.set_xlabel("epoch")
ax.set_ylabel("training loss")
ax.set_title("p-RRR training loss")
plt.tight_layout()
plt.show()

# Fit a linear RRR baseline on the same train/test split and rank

For comparison, we fit a standard linear reduced-rank regression at the same rank. Because linear RRR minimizes squared error and has no nonlinearity, its predictions can go negative — not ideal for spike counts.

In [ ]:
lin_rrr = LinearRRR(
    bias=True,
    rank=model_rank,
    regList=[reg_strength],
    zeromeanY=False,
    normstdY=False,
    verbose=False,
)
lin_rrr.fit([X_train], Y_train, zeromeanXs=[0], normstdXs=[0])

Y_pred_test_linear = lin_rrr.predict([X_test], unnormY=False)
print(f"Linear RRR test prediction shape: {Y_pred_test_linear.shape}")

In [ ]:
# Compare observed and predicted population means on the test set only.
Y_pred_test = pRRR.predict([X_test])
timestamps_test = timestamps[split_idx:]

mean_v1_obs_test = np.mean(Y_test, axis=1)
mean_v1_pred_test = np.mean(Y_pred_test, axis=1)
mean_v1_pred_test_linear = np.mean(Y_pred_test_linear, axis=1)
mean_aca_test = np.mean(X_test, axis=1)

f = plt.figure(figsize=(16, 5))
plt.plot(timestamps_test, mean_v1_obs_test, c='dodgerblue', label='V1 test (observed)')
plt.plot(timestamps_test, mean_aca_test, c='r', alpha=0.8, label='ACA test')
plt.plot(timestamps_test, mean_v1_pred_test, c='k', lw=2, label='V1 test predicted from ACA (p-RRR)')
plt.plot(timestamps_test, mean_v1_pred_test_linear, c='tab:green', lw=2, alpha=0.9, label='V1 test predicted from ACA (linear RRR)')
plt.xlim(timestamps_test[0], timestamps_test[min(len(timestamps_test) - 1, 30 * 35)])
plt.xlabel('time (s)')
plt.ylabel('mean population activity (spikes/bin)')
plt.legend()
plt.xlim(2600, 2640)
plt.show()

# Plot single-neuron test activity and predictions
Below we plot 10 randomly selected V1 neurons to inspect single-neuron prediction quality. D² (Poisson deviance explained) is the natural goodness-of-fit metric for count data; R² is shown for reference.

In [ ]:
n_neurons_to_plot = 10
rng = np.random.default_rng(6)

# Use the same section of time as before.
t_min, t_max = 2600, 2640
time_mask = (timestamps_test >= t_min) & (timestamps_test <= t_max)
t_plot = timestamps_test[time_mask]

if np.sum(time_mask) == 0:
    raise ValueError("No test timestamps found in the requested time window.")

# Random neuron indices for V1 population.
v1_indices = rng.choice(Y_test.shape[1], size=n_neurons_to_plot, replace=False)

fig, axes = plt.subplots(
    n_neurons_to_plot,
    1,
    figsize=(16, 2.1 * n_neurons_to_plot),
    sharex=True,
    constrained_layout=True,
    )

for row, v1_idx in enumerate(v1_indices):
    ax = axes[row]

    # Full test-series values for scoring.
    y_true_full = Y_test[:, v1_idx]
    y_pred_prrr_full = Y_pred_test[:, v1_idx]
    y_pred_linear_full = Y_pred_test_linear[:, v1_idx]

    # Windowed values for plotting.
    y_true = Y_test[time_mask, v1_idx]
    y_pred_prrr = Y_pred_test[time_mask, v1_idx]
    y_pred_linear = Y_pred_test_linear[time_mask, v1_idx]

    # D2 for Poisson requires strictly positive predictions.
    d2_prrr = sklearn.metrics.d2_tweedie_score(
        y_true_full,
        np.clip(y_pred_prrr_full, np.spacing(1), None),
        power=1,
    )
    d2_linear = sklearn.metrics.d2_tweedie_score(
        y_true_full,
        np.clip(y_pred_linear_full, np.spacing(1), None),
        power=1,
    )
    r2_prrr = sklearn.metrics.r2_score(y_true_full, y_pred_prrr_full)
    r2_linear = sklearn.metrics.r2_score(y_true_full, y_pred_linear_full)

    ax.plot(t_plot, y_true, c='dodgerblue', lw=1.2, label='V1 test (targets)' if row == 0 else None)
    ax.plot(t_plot, y_pred_prrr, c='k', lw=1.4, label='V1 test predicted from ACA (p-RRR)' if row == 0 else None)
    ax.plot(t_plot, y_pred_linear, c='tab:green', lw=1.4, alpha=0.95, label='V1 test predicted from ACA (linear RRR)' if row == 0 else None)
    ax.set_ylabel(f'cell {v1_idx}')
    ax.set_xlim(t_min, t_max)
    ax.set_title(
        f"p-RRR: D2={d2_prrr:.3f}, R2={r2_prrr:.3f} | "
        f"Linear: D2={d2_linear:.3f}, R2={r2_linear:.3f}",
        fontsize=12,
    )

axes[0].legend(loc='upper right', ncol=2, fontsize=9)
axes[-1].set_xlabel('time (s)')
fig.suptitle('Single-neuron test activity and predictions (10 random neurons)', y=1.01)
plt.show()